In [1]:
ORDERS = {
    101: {"status": "delivered", "total": 1200},
    102: {"status": "shipped", "total": 850},
    103: {"status": "processing", "total": 430},
}

In [18]:
def get_order_status(order_id):
    order = ORDERS.get(order_id, {"error": "order not found"})
    return order
get_order= get_order_status(101)
print(get_order)

{'status': 'delivered', 'total': 1200}


In [10]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_order_status",
            "description": "Returns the current status and total price of an order, given its order id.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "integer", "description": "The order's number, for example 101"}
                },
                "required": ["order_id"],
            },
        },
    }
]

In [17]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY"),
)

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": "তোমার নাম কী?"}],
    tools=tools,
)

message = response.choices[0].message
print(message.content)
print(message.tool_calls)

আমার নাম ChatGPT, আপনার ভার্চুয়াল সহকারী।
None


In [13]:
import json

call = message.tool_calls[0]
name = call.function.name
args = json.loads(call.function.arguments)
print(name)
print(args)

get_order_status
{'order_id': 102}


In [14]:
result = get_order_status(order_id=args["order_id"])
print(result)

{'status': 'shipped', 'total': 850}


In [15]:
messages = [
    {"role": "user", "content": "আমার order 102 এর অবস্থা কী?"},
    message,
    {
        "role": "tool",
        "tool_call_id": call.id,
        "content": json.dumps(result),
    },
]
print(len(messages))

3


In [16]:
final = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=messages,
    tools=tools,
)
print(final.choices[0].message.content)

অর্ডার ১০২ টি **শিপড** হয়েছে এবং মোট মূল্য **৳৮৫০**। যদি আরও কোনো তথ্য বা সহায়তা দরকার হয় জানাতে দ্বিধা করবেন না!


In [22]:
INVENTORY = {
    "laptop": 12,
    "mouse": 0,
    "keyboard": 5,
}

In [23]:
def get_inventory(product_name):
    inventory = INVENTORY.get(product_name)
    if inventory is None:
        return {"error": "product not found"}
    return {"product": product_name, "stock": inventory}

result = get_inventory("laptop")
print(result)

{'product': 'laptop', 'stock': 12}


In [24]:
print(get_inventory("monitor"))
print(get_inventory("mouse"))

{'error': 'product not found'}
{'product': 'mouse', 'stock': 0}
